# Décorateurs

Un décorateur est une fonction qui manipule une fonction ou une classe. C'est un élément du langage Python très pratique.

Nous allons voir dans ces travaux pratiques comment les utiliser.

## Décorateurs de fonctions : fonctions qui manipulent des fonctions

Écrivez une fonction `au_carre`, qui prend en entrée une fonction et qui renvoie cette même fonction dont le résultat est passé au carré :

- la fonction originale prend toujours deux arguments
- la fonction originale produit toujours un nombre
- la fonction au_carre prend en entrée et produit en sortie **une fonction**

In [ ]:
def additionne(a, b):
  return a + b


def soustrait(a, b):
  return a - b


def multiplie(a, b):
  return a * b


def au_carre(fonction):
  return fonction  # Votre code ici


print(au_carre(additionne)(2, 2))  # Doit valoir 16
print(au_carre(soustrait)(10, 5))  # Doit valoir 25
print(au_carre(multiplie)(3, 4))  # Doit valoir 144

### Conseils supplémentaires

`au_carre` définit une nouvelle fonction à deux arguments, qui appelle `fonction` et renvoie le carré de son résultat, puis renvoie cette nouvelle fonction, sans l'appeler. `au_carre(additionne)` est donc une fonction, et `au_carre(additionne)(2, 2)` l'appelle.

```python
def au_carre(fonction):
  def nouvelle_fonction(a, b):
    return ...

  return ...
```

### Solution

In [ ]:
def au_carre(fonction):
  def nouvelle_fonction(a, b):
    return fonction(a, b) ** 2

  return nouvelle_fonction


print(au_carre(additionne)(2, 2))
print(au_carre(soustrait)(10, 5))
print(au_carre(multiplie)(3, 4))

## Ordre des décorateurs empilés

Les deux décorateurs ci-dessous affichent un message avant et après l'appel de la fonction qu'ils décorent, et entourent son résultat d'une balise HTML.

Sans exécuter de code, écrivez dans l'ordre les lignes qu'affichera `print(salue("Ada"))`. Vérifiez ensuite votre réponse en exécutant les deux cellules.

In [ ]:
import functools


def gras(fonction):
  @functools.wraps(fonction)
  def nouvelle_fonction(*args, **kwargs):
    print("gras : avant")
    resultat = fonction(*args, **kwargs)
    print("gras : après")
    return f"<b>{resultat}</b>"

  return nouvelle_fonction


def italique(fonction):
  @functools.wraps(fonction)
  def nouvelle_fonction(*args, **kwargs):
    print("italique : avant")
    resultat = fonction(*args, **kwargs)
    print("italique : après")
    return f"<i>{resultat}</i>"

  return nouvelle_fonction


@gras
@italique
def salue(nom):
  print("salue")
  return f"Bonjour {nom}"

In [ ]:
print(salue("Ada"))

*Votre réponse*

### Conseils supplémentaires

`@gras` au-dessus de `@italique` équivaut à `salue = gras(italique(salue))`. Chaque décorateur affiche « avant », appelle la fonction qu'il enveloppe, puis affiche « après » : `print` n'affiche le résultat qu'une fois tous ces appels terminés.

```text
gras : avant
...
<b>...</b>
```

### Solution

```text
gras : avant
italique : avant
salue
italique : après
gras : après
<b><i>Bonjour Ada</i></b>
```

`@gras @italique def salue` équivaut à `salue = gras(italique(salue))` : `gras` enveloppe `italique`, qui enveloppe `salue`. L'appel traverse les couches de l'extérieur vers l'intérieur, et le résultat ressort dans l'ordre inverse.

## Mesure du temps d'exécution d'une fonction

Créez un décorateur `chronometre` qui affiche le temps d'exécution d'une fonction.

**Documentation utile**

- [`time.perf_counter`](https://docs.python.org/fr/3/library/time.html#time.perf_counter)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Relevez `time.perf_counter()` avant et après l'appel de la fonction : la durée en secondes est leur différence. La fonction interne accepte `*args` et `**kwargs`, et renvoie le résultat de la fonction décorée.

```python
import time


def chronometre(f):
  def nouvelle_f(*args, **kwargs):
    debut = ...
    resultat = ...
    print(...)
    return resultat

  return nouvelle_f
```

### Solution

In [ ]:
import time


def chronometre(f):
  def nouvelle_f(*args, **kwargs):
    debut = time.perf_counter()
    resultat = f(*args, **kwargs)
    print(f"Durée de '{f.__name__}' : {time.perf_counter() - debut} s")
    return resultat

  return nouvelle_f


@chronometre
def additionne(a, b):
  return a + b


additionne(1, 2)

## Registre de greffons

Un registre permet à une bibliothèque d'utiliser du code client qu'elle ne connaît pas, en trois étapes :

1. La bibliothèque met à disposition un décorateur, qui enregistre une classe ou une fonction.
2. Le code client applique ce décorateur.
3. La bibliothèque utilise les éléments enregistrés au moment voulu.

Les cellules suivantes créent un dossier `greffons/` avec deux modules de code client, `format_csv.py` et `format_json.py`, qui enregistrent chacun une classe d'export. Écrivez le côté bibliothèque :

1. Dans le module `registre.py`, complétez le décorateur de classe `exportateur(nom)`, qui enregistre la classe dans `exportateurs` sous la clef `nom`, et lève une `ValueError` si ce nom est déjà pris.
2. Écrivez une fonction `charge_greffons(dossier)`, qui importe tous les modules de `dossier`.
3. Affichez `registre.exportateurs` avant et après l'appel de `charge_greffons("greffons")`. Que remarquez-vous ?
4. Exportez `[[1, 2], [3, 4]]` en CSV en passant par le registre, puis vérifiez qu'enregistrer une autre classe sous le nom `"csv"` lève bien une `ValueError`.

**Documentation utile**

- [`pkgutil.iter_modules`](https://docs.python.org/fr/3/library/pkgutil.html#pkgutil.iter_modules)
- [`importlib.import_module`](https://docs.python.org/fr/3/library/importlib.html#importlib.import_module)

In [ ]:
from pathlib import Path

Path("greffons").mkdir(exist_ok=True)

In [ ]:
%%writefile greffons/format_csv.py
from registre import exportateur


@exportateur("csv")
class ExportateurCsv:
  def exporte(self, lignes):
    return "\n".join(",".join(map(str, ligne)) for ligne in lignes)

In [ ]:
%%writefile greffons/format_json.py
import json

from registre import exportateur


@exportateur("json")
class ExportateurJson:
  def exporte(self, lignes):
    return json.dumps(lignes)

In [ ]:
%%writefile registre.py
exportateurs = {}


def exportateur(nom):
  ...  # Votre code ici

In [ ]:
# Votre code ici

*Votre réponse*

### Conseils supplémentaires

`exportateur(nom)` renvoie un décorateur, qui reçoit la classe, la range dans `exportateurs` et la renvoie. `pkgutil.iter_modules([dossier])` parcourt les modules d'un dossier, chacun avec son attribut `nom`, et `importlib.import_module("greffons.format_csv")` importe un module d'après son nom. Avant l'import, le registre vaut `{}` ; après, il a les clefs `'csv'` et `'json'`. L'export donne deux lignes, `1,2` et `3,4`.

Python n'importe un module qu'une fois : après une correction de `registre.py`, redémarrez la session avant de réessayer.

Le module `registre.py` :

```python
exportateurs = {}


def exportateur(nom):
  def decorateur(cls):
    if ...:
      raise ValueError(...)
    exportateurs[nom] = ...
    return ...

  return ...
```

La suite, dans la cellule d'après :

```python
import importlib
import pkgutil

import registre


def charge_greffons(dossier):
  for module in pkgutil.iter_modules([dossier]):
    importlib.import_module(...)


print(registre.exportateurs)
charge_greffons("greffons")
print(registre.exportateurs)
print(registre.exportateurs["csv"]().exporte(...))

try:

  @registre.exportateur("csv")
  class AutreExportateur:
    pass

except ValueError as erreur:
  print(erreur)
```

### Solution

In [ ]:
%%writefile registre.py
exportateurs = {}


def exportateur(nom):
  def decorateur(cls):
    if nom in exportateurs:
      raise ValueError(f"Nom déjà pris : {nom}")
    exportateurs[nom] = cls
    return cls

  return decorateur

In [ ]:
import importlib
import pkgutil

import registre


def charge_greffons(dossier):
  for module in pkgutil.iter_modules([dossier]):
    importlib.import_module(f"{dossier}.{module.name}")


print(registre.exportateurs)
charge_greffons("greffons")
print(registre.exportateurs)
print(registre.exportateurs["csv"]().exporte([[1, 2], [3, 4]]))

try:

  @registre.exportateur("csv")
  class AutreExportateur:
    pass

except ValueError as erreur:
  print(erreur)

Le registre reste vide tant que les modules de `greffons/` ne sont pas importés : un décorateur s'exécute à la définition de la classe, donc à l'import de son module. Une bibliothèque qui repose sur un registre doit donc importer le code client avant de s'en servir : c'est le rôle de `charge_greffons`.

Ce patron s'appelle *Registry* en anglais.

## S'abonner à des événements

Une boutique veut déclencher plusieurs actions quand une commande est validée (envoyer une confirmation, mettre à jour le stock…), sans que le code qui valide la commande connaisse ces actions. C'est le patron *observateur*, écrit ici avec un décorateur.

Écrivez :

- un dictionnaire `abonnes`, qui associe à chaque événement la liste de ses fonctions abonnées ;
- un décorateur avec argument `abonne(evenement)`, qui ajoute la fonction décorée aux abonnées de `evenement` ;
- une fonction `publie(evenement, *args)`, qui appelle avec `args`, dans l'ordre d'abonnement, toutes les abonnées de `evenement`.

Abonnez deux fonctions à l'événement `"commande_validee"`, `envoie_confirmation` et `met_a_jour_stock`, puis publiez-le pour la commande `{"client": "Ada", "quantite": 2}` : les deux messages doivent s'afficher. Publier un événement sans abonné, comme `"commande_annulee"`, ne doit rien faire.

**Documentation utile**

- [`dict.setdefault`](https://docs.python.org/fr/3/library/stdtypes.html#dict.setdefault)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Comme `exportateur`, `abonne` renvoie un décorateur, qui renvoie la fonction telle quelle : elle reste appelable directement. `abonnes.setdefault(evenement, [])` renvoie la liste des abonnées, créée au besoin, et `abonnes.get(evenement, [])` évite une `KeyError` pour un événement sans abonné. Vous devez obtenir `Confirmation pour Ada`, puis `Stock diminué de 2`.

```python
abonnes = {}


def abonne(evenement):
  def decorateur(fonction):
    abonnes.setdefault(evenement, []).append(...)
    return ...

  return ...


def publie(evenement, *args):
  for fonction in ...:
    ...


@abonne("commande_validee")
def envoie_confirmation(commande):
  print("Confirmation pour", commande["client"])
```

### Solution

In [ ]:
abonnes = {}


def abonne(evenement):
  def decorateur(fonction):
    abonnes.setdefault(evenement, []).append(fonction)
    return fonction

  return decorateur


def publie(evenement, *args):
  for fonction in abonnes.get(evenement, []):
    fonction(*args)


@abonne("commande_validee")
def envoie_confirmation(commande):
  print("Confirmation pour", commande["client"])


@abonne("commande_validee")
def met_a_jour_stock(commande):
  print("Stock diminué de", commande["quantite"])


publie("commande_validee", {"client": "Ada", "quantite": 2})
publie("commande_annulee", {"client": "Ada"})

## Utilisation de `functools.wraps`

Décorez une fonction documentée avec votre `chronometre`, puis affichez son nom (attribut `__name__`) et sa documentation (attribut `__doc__`). Que remarquez-vous ?

Utilisez ensuite `functools.wraps` pour régler ce problème.

In [ ]:
# Votre code ici

*Votre réponse*

### Conseils supplémentaires

La fonction décorée est `nouvelle_f` : c'est son nom et sa documentation que vous lisez. `@functools.wraps(f)` se place juste au-dessus de `def nouvelle_f`.

```python
import functools
import time


def chronometre(f):
  @functools.wraps(...)
  def nouvelle_f(*args, **kwargs): ...

  return nouvelle_f


@chronometre
def additionne(a, b):
  """..."""
  return a + b


print(additionne.__name__, additionne.__doc__)
```

### Solution

In [ ]:
import functools
import time


def chronometre(f):
  @functools.wraps(f)
  def nouvelle_f(*args, **kwargs):
    debut = time.perf_counter()
    resultat = f(*args, **kwargs)
    print(f"Durée de '{f.__name__}' : {time.perf_counter() - debut} s")
    return resultat

  return nouvelle_f


@chronometre
def additionne(a, b):
  """Additionne deux entiers."""
  return a + b


print(additionne.__name__, additionne.__doc__)

Sans `functools.wraps`, `__name__` vaut le nom de la fonction interne (`nouvelle_f`) et `__doc__` vaut `None` : la fonction décorée a perdu son identité. Avec `functools.wraps`, elle retrouve son nom et sa documentation.

## Décorateur `chronometre` avec argument

Créez un décorateur `chronometre` qui prend un argument `unite` (`"s"` ou `"ms"`), l'unité dans laquelle afficher le temps d'exécution, et qui garde `functools.wraps`.

In [ ]:
# Votre code ici

### Conseils supplémentaires

Il y a trois niveaux de fonctions : `chronometre(unite)` renvoie `decorateur`, qui reçoit la fonction et renvoie `nouvelle_f`. `@chronometre(unite="ms")` appelle d'abord `chronometre`, puis applique ce qu'elle renvoie à la fonction décorée. Une durée en secondes multipliée par 1000 donne des millisecondes.

```python
import functools
import time


def chronometre(unite="s"):
  def decorateur(fonction):
    @functools.wraps(fonction)
    def nouvelle_f(*args, **kwargs):
      debut = time.perf_counter()
      resultat = ...
      duree = ...
      if unite == "ms":
        ...
      print(...)
      return resultat

    return ...

  return ...
```

### Solution

In [ ]:
import functools
import time


def chronometre(unite="s"):
  if unite not in ("s", "ms"):
    raise ValueError("L'unité doit être s ou ms")

  def decorateur(fonction):
    @functools.wraps(fonction)
    def nouvelle_f(*args, **kwargs):
      debut = time.perf_counter()
      resultat = fonction(*args, **kwargs)
      duree = time.perf_counter() - debut
      if unite == "ms":
        duree *= 1000
      print(f"Durée de '{fonction.__name__}' : {duree} {unite}")
      return resultat

    return nouvelle_f

  return decorateur


@chronometre(unite="ms")
def additionne(a, b):
  """Additionne deux entiers."""
  return a + b


print(additionne.__name__, additionne.__doc__)

## Mise en cache des résultats

Écrivez un décorateur `memoise` qui mémorise dans un dictionnaire le résultat de chaque appel, avec ses arguments pour clef, et le renvoie directement quand la fonction est rappelée avec les mêmes arguments.

Comparez ensuite, avec votre décorateur `chronometre`, la durée de `fibonacci(32)` sans cache, avec `memoise`, et avec `functools.cache`. Pour cela :

- écrivez trois fonctions récursives distinctes, chacune s'appelant elle-même, la première sans décorateur, les deux autres décorées avec `@` ;
- ne mesurez que l'appel extérieur, avec `chronometre(unite="ms")(f)(32)`, et n'appliquez pas `@chronometre` à une fonction récursive.

**Documentation utile**

- [`functools.cache`](https://docs.python.org/fr/3/library/functools.html#functools.cache)

In [ ]:
def fibonacci(n: int) -> int:
  return n if n < 2 else fibonacci(n - 1) + fibonacci(n - 2)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Le dictionnaire se crée dans `memoise` : la fonction interne le garde par fermeture, d'un appel à l'autre. `args` est un n-uplet, donc hachable : il sert directement de clef. `fibonacci(32)` vaut 2178309.

```python
import functools


def memoise(fonction):
  cache = {}

  @functools.wraps(fonction)
  def nouvelle_fonction(*args):
    if ...:
      cache[args] = ...
    return ...

  return nouvelle_fonction


@memoise
def fibonacci(n: int) -> int:
  return n if n < 2 else fibonacci(n - 1) + fibonacci(n - 2)
```

### Solution

In [ ]:
import functools


def memoise(fonction):
  cache = {}

  @functools.wraps(fonction)
  def nouvelle_fonction(*args):
    if args not in cache:
      cache[args] = fonction(*args)
    return cache[args]

  return nouvelle_fonction


def fibonacci_sans_cache(n: int) -> int:
  return n if n < 2 else fibonacci_sans_cache(n - 1) + fibonacci_sans_cache(n - 2)


@memoise
def fibonacci(n: int) -> int:
  return n if n < 2 else fibonacci(n - 1) + fibonacci(n - 2)


@functools.cache
def fibonacci_cache(n: int) -> int:
  return n if n < 2 else fibonacci_cache(n - 1) + fibonacci_cache(n - 2)


for f in (fibonacci_sans_cache, fibonacci, fibonacci_cache):
  chronometre(unite="ms")(f)(32)

Sans cache, `fibonacci(32)` recalcule les mêmes valeurs des millions de fois ; avec, chaque valeur n'est calculée qu'une fois. `functools.cache` fait la même chose que `memoise`, en gérant aussi les arguments nommés, et fournit `cache_info()` et `cache_clear()`.

Deux pièges tiennent à ce que `@` réaffecte le nom de la fonction :

- `@chronometre` sur une fonction récursive décore aussi ses appels récursifs, qui passent par ce nom : `fibonacci(32)` afficherait une ligne pour chacun de ses 7 049 155 appels. D'où `chronometre(unite="ms")(f)(32)`, qui ne mesure que l'appel extérieur.
- `memoise(fibonacci)(32)`, sans réaffecter le nom, ne gagne presque rien : à l'intérieur, la récursion appelle toujours la fonction d'origine, sans cache. C'est pourquoi la fonction est décorée avec `@`.